In [1]:
import anndata as ad 
import squidpy as sq 
import numpy as np 
import pandas as pd


KeyboardInterrupt: 

In [4]:
adata = ad.read_h5ad("/home/boyesh/IMLAkoyafusion/prototype/phenotyping/slide_1_052024 P7HuP120 #03 SG03_phenotyped.h5ad")

In [5]:
adata.uns.keys()

dict_keys(['leiden', 'leiden_colors', 'neighbors', 'pca', 'umap'])

In [6]:
sq.gr.spatial_neighbors(adata, n_neighs=10, coord_type='generic', copy=False)
adata.uns.keys()


INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        


dict_keys(['leiden', 'leiden_colors', 'neighbors', 'pca', 'umap', 'spatial_neighbors'])

In [7]:
sq.gr.nhood_enrichment(adata, cluster_key="leiden", n_perms=100, seed=1234, copy=False)
print(adata.uns["leiden_nhood_enrichment"].keys())


100%|██████████| 100/100 [00:01<00:00, 77.91/s]

dict_keys(['zscore', 'count'])


In [8]:
sq.gr.co_occurrence(adata, cluster_key="leiden", spatial_key='spatial', interval=np.arange(30, 330, 30), copy=False)
print(type(adata.uns["leiden_co_occurrence"]))
print(adata.uns["leiden_co_occurrence"].keys())

<class 'dict'>
dict_keys(['occ', 'interval'])


In [9]:
print(adata.uns["leiden_co_occurrence"]["occ"].shape)
print(adata.uns["leiden_co_occurrence"]["interval"])


(15, 15, 9)
[ 30.  60.  90. 120. 150. 180. 210. 240. 270. 300.]


In [18]:
intervals = adata.uns["leiden_co_occurrence"]["interval"][:-1]
co_occ_df = pd.DataFrame()
leiden_labels = adata.obs['leiden'].unique()
for idx, interval_val in enumerate(intervals):
    df = pd.DataFrame(adata.uns["leiden_co_occurrence"]["occ"][:, :, idx], index=leiden_labels, columns=leiden_labels)
    df["interval"] = interval_val
    co_occ_df = pd.concat([co_occ_df, df])
        

In [19]:
print(co_occ_df.columns.tolist())
print(co_occ_df.head())

['0', '13', '10', '4', '6', '9', '2', '7', '12', '3', '5', '8', '14', '11', '1', 'interval']
           0         13        10         4         6         9         2  \
0   6.814252   2.397212  0.302504  0.409144  3.551691  0.765089  2.526709   
13  2.397212  18.753949  0.001851  0.007830  0.723140  0.247830  0.999541   
10  0.302504   0.001851  2.455711  1.842143  0.267177  0.311712  0.276514   
4   0.409144   0.007830  1.842143  1.994785  0.532905  0.578679  0.432577   
6   3.551691   0.723140  0.267177  0.532905  5.069979  1.850608  3.198833   

           7        12         3         5         8        14        11  \
0   0.830222  0.680336  0.486009  0.280421  1.000511  0.441058  0.032449   
13  0.293026  0.126718  0.437613  0.781895  0.840847  0.021230  0.078393   
10  0.474922  1.182191  0.300364  0.085929  0.625419  1.322941  0.008601   
4   0.792179  1.132984  0.546877  0.179411  0.643291  1.658221  0.022756   
6   1.804453  0.898205  0.432556  0.259775  1.213379  0.718142  

In [20]:
co_occ_df = pd.melt(co_occ_df, id_vars="interval", value_vars=leiden_labels, value_name="score", var_name="cluster_1")
print(co_occ_df.head())
print(co_occ_df.columns.tolist())

   interval cluster_1     score
0      30.0         0  6.814252
1      30.0         0  2.397212
2      30.0         0  0.302504
3      30.0         0  0.409144
4      30.0         0  3.551691
['interval', 'cluster_1', 'score']


In [21]:
sq.gr.ripley(adata, cluster_key='leiden', mode='L', n_simulations=100, n_neigh=10, seed=1234, copy=False)
print(type(adata.uns["leiden_ripley_L"]))
print(adata.uns["leiden_ripley_L"].keys())

<class 'dict'>
dict_keys(['L_stat', 'sims_stat', 'bins', 'pvalues'])


In [22]:
r = adata.uns["leiden_ripley_L"]
print(type(r["L_stat"]))
print(type(r["bins"]))
print(type(r["pvalues"]))
print(r["L_stat"].shape if hasattr(r["L_stat"], 'shape') else r["L_stat"])
print(r["bins"].shape if hasattr(r["bins"], 'shape') else r["bins"])
print(r["pvalues"].shape if hasattr(r["pvalues"], 'shape') else r["pvalues"])

<class 'pandas.core.frame.DataFrame'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(750, 3)
(50,)
(15, 50)


In [23]:
print(r["L_stat"].head())
print(r["L_stat"].columns.tolist())

          bins leiden       stats
0     0.000000      0    0.000000
1   428.244672      0  124.423359
2   856.489344      0  193.330503
3  1284.734016      0  236.179336
4  1712.978688      0  266.409253
['bins', 'leiden', 'stats']
